# GNN Inference Run — Jetson RF Map Reconstruction

Loads the `.pth` trained in `GNN_patch_and_parquet_ver.ipynb` and checks whether the model can reconstruct a full 256x256 RF (path-loss) map from a sparse set of samples on the Jetson.

**Inputs per Test File** `.parquet`: `building_mask`, `tx_origin`, `path_loss` (each a flat 256*256 array).

**Pipeline per Sample**
1. Build the 3-channel input, where the sparse channel keeps 1–10% of the true path-loss pixels.
2. Run the model and de-normalize the output back to dB.
3. Compare against the full ground-truth map (RMSE / MAE in dB, % pixels within tolerance) and time the inference.

**File Directory**
- Test files: `data/gnn_inference_test/` (drop `.parquet` files here)
- Weights: `benchmarks/GNN_no_tx.pth` (change `WEIGHTS_PATH` below if needed)
- Output figures + CSV: `models/GNN/inference_outputs/`

Requirements on the Jetson: `torch` (JetPack build), `numpy`, `pandas`, `pyarrow`, `matplotlib`.

In [ ]:
import os, glob, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def jetson_info():
    try:
        with open("/etc/nv_tegra_release") as f:
            return f.readline().strip()
    except OSError:
        return None

print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
print(f"Jetson  : {jetson_info() or 'not detected'}")

### Path Configuration

In [ ]:
# Resolve the repo root by walking up from the notebook's folder for data/ and models/.
def find_repo_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, "data")) and os.path.isdir(os.path.join(d, "models")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError("Could not locate repo root (folder containing data/ and models/).")
        d = parent

REPO_ROOT    = find_repo_root()
WEIGHTS_PATH = os.path.join(REPO_ROOT, "benchmarks", "GNN_no_tx.pth")
TEST_DIR     = os.path.join(REPO_ROOT, "data", "gnn_inference_test")
OUT_DIR      = os.path.join(REPO_ROOT, "models", "GNN", "inference_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# --- Data constants (must match training) ------------------------------
H = W = 256
MIN_DB, MAX_DB = 30.0, 140.0
NUM_SAMPLES_LOW, NUM_SAMPLES_HIGH = 655, 6553   # ~1%-10% of pixels kept as sparse measurements

# Set to False to zero the tx_origin channel (e.g. if this checkpoint was trained without TX location).
USE_TX_CHANNEL = True

# --- Run settings ------------------------------------------------------
SEED         = 0          # fixed so the sparse sample pattern is reproducible
MAX_FILES    = None       # None = every parquet in TEST_DIR
WARMUP_ITERS = 5          # GPU warm-up before timing (first CUDA calls are slow on Jetson)
TIMED_ITERS  = 20         # timed forward passes per sample
USE_FP16     = False      # try True on Jetson for speed; compare accuracy against fp32
TOL_DB       = 6.0        # a pixel counts as "correct" if |pred - gt| <= TOL_DB dB
SAVE_FIGS    = True

print("Weights :", WEIGHTS_PATH, "| exists:", os.path.isfile(WEIGHTS_PATH))
print("Test dir:", TEST_DIR)
print("Out dir :", OUT_DIR)

### Model definition

In [ ]:
def knn_graph(x, k):
    # x: (B, C, N, 1) -> (B, N, k) neighbor indices in feature space
    B, C, N, _ = x.shape
    x_sq = x.squeeze(-1).permute(0, 2, 1)
    dist = torch.cdist(x_sq, x_sq, p=2)
    _, idx = dist.topk(k + 1, dim=-1, largest=False)
    return idx[:, :, 1:]  # drop self-loop


class DynConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, k=9, dilation=1, act=None):
        super().__init__()
        self.k = k
        self.conv = nn.Conv2d(in_channels * 2, out_channels, 1)
        self.act = act

    def forward(self, x):
        B, C, N, _ = x.shape
        idx = knn_graph(x, self.k)
        x_flat = x.squeeze(-1).permute(0, 2, 1)
        idx_exp = idx.unsqueeze(-1).expand(-1, -1, -1, C)
        neighbors = torch.gather(x_flat.unsqueeze(2).expand(-1, -1, self.k, -1), 1, idx_exp)
        center = x_flat.unsqueeze(2).expand_as(neighbors)
        edge = torch.cat([center, neighbors - center], dim=-1)
        agg = edge.max(dim=2).values.permute(0, 2, 1).unsqueeze(-1)
        out = self.conv(agg)
        if self.act is not None:
            out = self.act(out)
        return out


class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = (keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)).floor_()
        return x.div(keep_prob) * random_tensor


class GrapherModule(nn.Module):
    def __init__(self, in_channels, hidden_channels, k=9, dilation=1, drop_path=0.0):
        super().__init__()
        self.fc1 = nn.Sequential(nn.Conv2d(in_channels, in_channels, 1), nn.BatchNorm2d(in_channels))
        self.graph_conv = nn.Sequential(
            DynConv2d(in_channels, hidden_channels, k, dilation, act=None),
            nn.BatchNorm2d(hidden_channels),
            nn.GELU(),
        )
        self.fc2 = nn.Sequential(nn.Conv2d(hidden_channels, in_channels, 1), nn.BatchNorm2d(in_channels))
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x):
        shortcut = x
        x = self.fc2(self.graph_conv(self.fc1(x)))
        return self.drop_path(x) + shortcut


class FFNModule(nn.Module):
    def __init__(self, in_channels, hidden_channels, drop_path=0.0):
        super().__init__()
        self.fc1 = nn.Sequential(nn.Conv2d(in_channels, hidden_channels, 1), nn.BatchNorm2d(hidden_channels), nn.GELU())
        self.fc2 = nn.Sequential(nn.Conv2d(hidden_channels, in_channels, 1), nn.BatchNorm2d(in_channels))
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x):
        shortcut = x
        x = self.fc2(self.fc1(x))
        return self.drop_path(x) + shortcut


class ViGBlock(nn.Module):
    def __init__(self, channels, k=9, dilation=1, drop_path=0.0):
        super().__init__()
        self.grapher = GrapherModule(channels, channels * 2, k, dilation, drop_path)
        self.ffn = FFNModule(channels, channels * 4, drop_path)

    def forward(self, x):
        return self.ffn(self.grapher(x))


class PatchEmbed(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size=8):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, E, n_h, n_w = x.shape
        x = self.norm(x.flatten(2).transpose(1, 2))
        return x.transpose(1, 2).reshape(B, E, n_h, n_w)


class PatchUnembed(nn.Module):
    def __init__(self, embed_dim, out_channels, patch_size=8):
        super().__init__()
        self.proj = nn.ConvTranspose2d(embed_dim, out_channels, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x)


class ViGPatchModel(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, embed_dim=64, num_blocks=4,
                 patch_size=8, k=9, drop_path=0.2):
        super().__init__()
        self.patch_size = patch_size
        self.patch_embed = PatchEmbed(in_channels, embed_dim, patch_size)
        dpr = [x.item() for x in torch.linspace(0, drop_path, num_blocks)]
        self.blocks = nn.ModuleList([ViGBlock(embed_dim, k=k, dilation=1, drop_path=dpr[i]) for i in range(num_blocks)])
        self.patch_unembed = PatchUnembed(embed_dim, out_channels, patch_size)
        self.norm = nn.BatchNorm2d(embed_dim)

    def _pad_to_patch(self, x):
        B, C, H, W = x.shape
        ph = (self.patch_size - H % self.patch_size) % self.patch_size
        pw = (self.patch_size - W % self.patch_size) % self.patch_size
        if ph > 0 or pw > 0:
            x = F.pad(x, (0, pw, 0, ph))
        return x, H, W

    def forward(self, x, mask=None):
        x, orig_H, orig_W = self._pad_to_patch(x)
        x = self.patch_embed(x)
        B, E, n_h, n_w = x.shape
        x = x.reshape(B, E, n_h * n_w, 1)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x.reshape(B, E, n_h, n_w))
        x = self.patch_unembed(x)
        return x[:, :, :orig_H, :orig_W]

## Load trained weights

In [ ]:
model = ViGPatchModel(in_channels=3, out_channels=1, embed_dim=64, num_blocks=4,
                      patch_size=8, k=9, drop_path=0.1)

try:
    state = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=True)
except TypeError:  # older JetPack PyTorch builds without weights_only
    state = torch.load(WEIGHTS_PATH, map_location="cpu")
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]
state = {k.replace("module.", "", 1): v for k, v in state.items()}  # strip DataParallel prefix if present

model.load_state_dict(state, strict=True)
model = model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {os.path.basename(WEIGHTS_PATH)} | params: {n_params:,} | device: {device}")

## Test data loader

In [ ]:
def load_parquet_sample(path, rng):
    df = pd.read_parquet(path, engine="pyarrow")
    row = df.iloc[0]
    env_map = np.asarray(row["building_mask"], dtype=np.float32).reshape(H, W)
    tx_map  = np.asarray(row["tx_origin"],     dtype=np.float32).reshape(H, W)
    has_gt  = "path_loss" in df.columns
    radio_map = np.asarray(row["path_loss"], dtype=np.float32).reshape(H, W) if has_gt else None

    if not USE_TX_CHANNEL:
        tx_map = np.zeros_like(tx_map)

    if "sparse_samples" in df.columns:
        # Real measurements supplied directly (dB, 0 = not measured)
        sparse = np.asarray(row["sparse_samples"], dtype=np.float32).reshape(H, W)
    elif has_gt:
        # Simulate sparse sensor measurements from the ground-truth map (same as training)
        n = rng.integers(NUM_SAMPLES_LOW, NUM_SAMPLES_HIGH)
        xs = rng.integers(0, W, size=n)
        ys = rng.integers(0, H, size=n)
        sparse = np.zeros((H, W), dtype=np.float32)
        sparse[xs, ys] = radio_map[xs, ys]
    else:
        raise ValueError(f"{os.path.basename(path)} has neither 'path_loss' nor 'sparse_samples'.")

    sparse_norm = np.zeros_like(sparse)
    valid = sparse > 0
    sparse_norm[valid] = (sparse[valid] - MIN_DB) / (MAX_DB - MIN_DB)
    sparse_norm = np.clip(sparse_norm, 0.0, 1.0)

    x = torch.from_numpy(np.stack([env_map, tx_map, sparse_norm], axis=0)).float().unsqueeze(0)
    return {
        "name": os.path.basename(path), "x": x, "env": env_map, "tx": tx_map,
        "sparse_db": np.where(valid, sparse, np.nan), "n_samples": int(valid.sum()),
        "gt_db": radio_map,
    }

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.parquet")))
if MAX_FILES:
    test_files = test_files[:MAX_FILES]
print(f"Found {len(test_files)} parquet file(s) in {TEST_DIR}")
if not test_files:
    print("-> Add .parquet test files to data/gnn_inference_test/ and re-run this cell.")

### Inference + Timing

In [ ]:
def denorm_db(y):
    return y * (MAX_DB - MIN_DB) + MIN_DB

def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

@torch.no_grad()
def run_model(x):
    if USE_FP16 and device.type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            return model(x).float()
    return model(x)

rng = np.random.default_rng(SEED)
results, samples = [], []

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

for i, path in enumerate(test_files):
    s = load_parquet_sample(path, rng)
    x = s["x"].to(device)

    if i == 0:  # warm up CUDA kernels once before timing
        for _ in range(WARMUP_ITERS):
            run_model(x)
        sync()

    times = []
    for _ in range(TIMED_ITERS):
        sync(); t0 = time.perf_counter()
        y = run_model(x)
        sync(); times.append((time.perf_counter() - t0) * 1000)

    pred_db = denorm_db(y.squeeze().cpu().numpy())
    s["pred_db"] = pred_db
    rec = {"file": s["name"], "n_sparse": s["n_samples"],
           "sparse_pct": 100 * s["n_samples"] / (H * W),
           "latency_ms_mean": float(np.mean(times)), "latency_ms_p95": float(np.percentile(times, 95))}

    if s["gt_db"] is not None:
        err = pred_db - s["gt_db"]
        rec.update({
            "rmse_db": float(np.sqrt(np.mean(err ** 2))),
            "mae_db": float(np.mean(np.abs(err))),
            f"acc_within_{TOL_DB:g}dB_pct": float(100 * np.mean(np.abs(err) <= TOL_DB)),
        })
    results.append(rec)
    samples.append(s)
    print(f"[{i+1}/{len(test_files)}] {s['name']}: " +
          ", ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in rec.items() if k != "file"))

if device.type == "cuda" and test_files:
    print(f"\nPeak GPU memory: {torch.cuda.max_memory_allocated() / 1024**2:.1f} MB")

## Summary

In [ ]:
if results:
    df_res = pd.DataFrame(results)
    display(df_res.round(3))
    print("\nMean over all test files:")
    print(df_res.drop(columns=["file"]).mean(numeric_only=True).round(3).to_string())
    print(f"\nThroughput: {1000 / df_res['latency_ms_mean'].mean():.1f} maps/s "
          f"({'fp16' if USE_FP16 else 'fp32'}, batch=1, {device})")
    csv_path = os.path.join(OUT_DIR, "gnn_inference_results.csv")
    df_res.to_csv(csv_path, index=False)
    print("Saved", csv_path)

## Visualize Reconstructed RF maps

In [ ]:
def plot_sample(s):
    has_gt = s["gt_db"] is not None
    ncols = 5 if has_gt else 3
    fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4))
    vmin = np.nanmin(s["gt_db"]) if has_gt else np.nanmin(s["pred_db"])
    vmax = np.nanmax(s["gt_db"]) if has_gt else np.nanmax(s["pred_db"])

    axes[0].imshow(s["env"], cmap="gray"); axes[0].set_title("Building mask")
    if s["tx"].max() > 0:
        ty, tx = np.unravel_index(np.argmax(s["tx"]), s["tx"].shape)
        axes[0].plot(tx, ty, "r*", markersize=12, label="TX"); axes[0].legend(loc="upper right")

    axes[1].imshow(np.zeros_like(s["env"]), cmap="gray", vmin=0, vmax=1)
    axes[1].imshow(s["sparse_db"], cmap="viridis", vmin=vmin, vmax=vmax, interpolation="nearest")
    axes[1].set_title(f"Sparse input ({s['n_samples']} px)")

    im = axes[2].imshow(s["pred_db"], cmap="viridis", vmin=vmin, vmax=vmax)
    axes[2].set_title("GNN reconstruction (dB)")

    if has_gt:
        axes[3].imshow(s["gt_db"], cmap="viridis", vmin=vmin, vmax=vmax)
        axes[3].set_title("Ground truth (dB)")
        err = np.abs(s["pred_db"] - s["gt_db"])
        im_e = axes[4].imshow(err, cmap="magma")
        axes[4].set_title(f"|Error| (RMSE {np.sqrt(np.mean(err**2)):.2f} dB)")
        fig.colorbar(im_e, ax=axes[4], fraction=0.046)
    fig.colorbar(im, ax=axes[2], fraction=0.046)

    for a in axes: a.axis("off")
    fig.suptitle(s["name"]); fig.tight_layout()
    if SAVE_FIGS:
        fig.savefig(os.path.join(OUT_DIR, os.path.splitext(s["name"])[0] + "_rfmap.png"), dpi=120)
    plt.show()

for s in samples:
    plot_sample(s)